# FiftyOne curation loop — see, mark, act, verify

The complete dataset-curation workflow on `ppe-raw`:
1. **Load** the dataset (import from YOLO folders if needed)
2. **Launch** the app in this notebook
3. **GUI**: review images, tag bad ones `bad-label`
4. **Read** the tags back in code
5. **Act**: export a clean dataset excluding them
6. **Verify** the export like an engineer

Principle: **GUI for judgment, code for lineage.**

## Cell 1 — load (or import) the dataset

Never trust a cached empty dataset — a failed prior import must not be reused.

In [1]:
import fiftyone as fo
from pathlib import Path

NAME = "ppe-raw"
BASE = Path("../data/raw/css-v30")          # notebook lives in notebooks/, hence ..
YAML = Path("../data/css-v30-local.yaml")   # corrected yaml (created by curate_dataset.py)

if fo.dataset_exists(NAME) and len(fo.load_dataset(NAME)) == 0:
    fo.delete_dataset(NAME)                  # corrupt/empty cache -> rebuild

if fo.dataset_exists(NAME):
    ds = fo.load_dataset(NAME)
else:
    ds = fo.Dataset(NAME, persistent=True)
    for split_dir, yaml_key in [("train", "train"), ("valid", "val"), ("test", "test")]:
        ds.add_dir(
            dataset_dir=str(BASE),
            dataset_type=fo.types.YOLOv5Dataset,
            split=yaml_key,
            tags=split_dir,
            yaml_path=str(YAML),
        )

print(ds.name, len(ds), "samples")           # expect: ppe-raw 717 samples

c:\Users\rajaa.RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ppe-raw 717 samples


## Cell 2 — launch the app inside the notebook

Same process as this code, so the persistence bug can't bite. Pop the app out with the expand icon if cramped.

In [2]:
session = fo.launch_app(ds)

## Now: the GUI part (judgment, ~10 min)

In the app above: click an image, flip through ~20 with `<` / `>`.

For each genuinely bad one — box not hugging the object, missed person, wrong class, mangled augmented boxes — click the **tag icon** in the expanded view, type `bad-label`, Enter.

Be picky: 3–8 tags is a normal haul. Then continue below.

## Cell 3 — read your GUI decisions from code

The bridge: GUI clicks became queryable data.

In [4]:
bad_imgs = ds.match_tags("bad-label")            # whole images you tagged
bad_boxes = ds.match_labels(tags="bad-label")    # images containing a tagged BOX
print(f"sample-tagged: {len(bad_imgs)}, label-tagged: {len(bad_boxes)}")

from pathlib import Path
for s in bad_boxes:
    flagged = [d.label for d in s.ground_truth.detections if "bad-label" in d.tags]
    print(" ", Path(s.filepath).name, "→ bad boxes:", flagged)

sample-tagged: 0, label-tagged: 1
  2008_008753_jpg.rf.8d06fcf951f7e512797e9a30ad3f8747.jpg → bad boxes: ['Person']


## Cell 4 — act: export everything *except* the bad images

`bool=False` inverts the tag match. Export writes a brand-new YOLO dataset; the originals are untouched.

In [5]:
clean = ds.match_tags("bad-label", bool=False)
clean.export(
    export_dir="../data/scratch-clean-export",
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
)
print("exported", len(clean), "of", len(ds))

 100% |█████████████████| 717/717 [7.6s elapsed, 0s remaining, 65.9 samples/s]       
exported 717 of 717


## Cell 5 — verify, like an engineer

Never trust an export you didn't count. Both checks must pass:
- files on disk == Cell 4's exported count
- tagged images that leaked into the export == **0**

In [6]:
exported = list(Path("../data/scratch-clean-export").rglob("*.jpg"))
print(len(exported), "files on disk")        # must equal Cell 4's count
bad_names = {Path(s.filepath).name for s in bad}
leaked = [p for p in exported if p.name in bad_names]
print("tagged images that leaked into export:", len(leaked))   # must be 0

717 files on disk
tagged images that leaked into export: 0


## Cleanup

The scratch export was just for practice — remove it. (Uncomment and run.)

In [ ]:
# import shutil
# shutil.rmtree("../data/scratch-clean-export")
# print("scratch export removed")